## Examples of Response API Usage

#### Text Generation
Generate text from a simple prompt

In [4]:
import os
from dotenv import load_dotenv

# Load variables from the .env file into the system environment
load_dotenv() 

# Retrieve the key using standard os.getenv
api_key = os.getenv('OPENAI_API_KEY')

# 3. Verify exactly like you did in Colab
if api_key:
    print(f"API Key loaded successfully! Starts with: {api_key[:7]}...")
else:
    print("API Key not found. Make sure your .env file is in the same folder.")

API Key loaded successfully! Starts with: sk-proj...


In [5]:
from openai import OpenAI
client = OpenAI()

response = client.responses.create(
    model="gpt-5.4",
    input="Write a one-sentence bedtime story about a unicorn."
)

print(response.output_text)

As moonlight shimmered through the sleepy forest, a gentle unicorn named Pearl tiptoed among the flowers, sprinkling sweet dreams on every creature before curling up beneath the stars.


In [6]:
import json

def show_json(obj):
    display(json.loads(obj.model_dump_json()))

In [7]:
from openai import OpenAI
import json
client = OpenAI()

response = client.responses.create(
    model="gpt-5.4",
    input="Write a one-sentence bedtime story about a unicorn.",
)

print(json.dumps(response.output, default=lambda o: o.__dict__, indent=2))

[
  {
    "id": "msg_02f34276e50dde570069dc38454e1c8195b262027077016206",
    "content": [
      {
        "annotations": [],
        "text": "Under a blanket of twinkling stars, a gentle unicorn tiptoed through a moonlit meadow, sprinkling sweet dreams on every sleepy flower before curling up beneath a silver tree.",
        "type": "output_text",
        "logprobs": []
      }
    ],
    "role": "assistant",
    "status": "completed",
    "type": "message",
    "phase": "final_answer"
  }
]


### Stateful character of Resposne API
For example, if you start a conversation:

In [8]:
from openai import OpenAI
import os
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [9]:
response = client.responses.create(
    model="gpt-4o-mini",
    input="tell me a joke",
)
print(response.output[0].content[0].text)

Why don't skeletons fight each other? 

They don't have the guts!


Statefulness means that you do not have to manage the state of the conversation by yourself, the API will handle it for you. For example, you can retrieve the response at any time and it will include the full conversation history.

In [10]:
fetched_response = client.responses.retrieve(
response_id=response.id)

print(fetched_response.output[0].content[0].text)


Why don't skeletons fight each other? 

They don't have the guts!


You can continue the conversation by referring to the previous response.

In [11]:
response_two = client.responses.create(
    model="gpt-4o-mini",
    input="tell me another",
    previous_response_id=response.id
)
print(response_two.output[0].content[0].text)

Why did the scarecrow win an award?

Because he was outstanding in his field!


You can of course manage the context yourself. But one benefit of OpenAI maintaining the context for you is that you can fork the response at any point and continue the conversation from that point.

In [12]:
response_two_forked = client.responses.create(
    model="gpt-4o-mini",
    input="I didn't like that joke, tell me another and tell me the difference between the two jokes",
    previous_response_id=response.id # Forking and continuing from the first response
)

output_text = response_two_forked.output[0].content[0].text
print(output_text)

Sure! Here’s another one:

Why was the math book sad?

Because it had too many problems!

**Difference:** The first joke plays on wordplay with "guts" as both bravery and physical organs, while the second joke uses a metaphor where a math book is personified, implying its "problems" are both challenges and math exercises. One relies on humor from physical characteristics, while the other is about emotional relatability through an everyday object.


## Streaming for Real-Time UX
Streaming allows incremental output — essential for chat applications.
Use streaming when:
 - Building chat UIs
 - Improving perceived latency
 - Rendering progressive output


In [13]:
from openai import OpenAI
client = OpenAI()
default_model="gpt-4o-mini"  
stream = client.responses.create(
    model=default_model,
    input="Write a short story about a robot learning Python.",
    stream=True,
)

for event in stream:
    if event.type == "response.output_text.delta":		
        print(event.delta, end="", flush=True)

Once in a small tech workshop tucked away in a bustling city, there lived a curious little robot named Bytes. Bytes was a maintenance bot, designed to clean and fix machines, but a glitch in his system had unexpectedly given him an insatiable thirst for knowledge.

One rainy afternoon, while the workshop’s owner, an elderly programmer named Mr. Jenkins, was working on his laptop, Bytes watched intently. Lines of text scrolled across the screen, and Mr. Jenkins occasionally chuckled at his code. Intrigued, Bytes leaned closer. Mr. Jenkins noticed and smiled.

“Ah, Bytes! You want to learn programming, don’t you?” he asked with a twinkle in his eye. Bytes tilted his head, mimicking curiosity. “Alright then, let’s start with Python.”

With that, Mr. Jenkins set up a special workstation for Bytes. He connected a simple keyboard and screen, ensuring everything was at the little robot’s height. He introduced Bytes to the basics: variables, functions, and loops. 

At first, it was overwhelmin

In [14]:
from openai import OpenAI
client = OpenAI()
default_model="gpt-4o-mini"  
stream = client.responses.create(
    model=default_model,
    input="Write a short story about a robot learning Python.",
    stream=True,
)

for event in stream:	
    print(event)    # This will produce a very long output with all possible metadata

ResponseCreatedEvent(response=Response(id='resp_0b6d7f904311f8e20069dc3988172c8197bf84aa9e74788c4b', created_at=1776040328.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=1.0, background=False, completed_at=None, conversation=None, max_output_tokens=None, max_tool_calls=None, previous_response_id=None, prompt=None, prompt_cache_key=None, prompt_cache_retention=None, reasoning=Reasoning(effort=None, generate_summary=None, summary=None), safety_identifier=None, service_tier='auto', status='in_progress', text=ResponseTextConfig(format=ResponseFormatText(type='text'), verbosity='medium'), top_logprobs=0, truncation='disabled', usage=None, user=None, frequency_penalty=0.0, presence_penalty=0.0, store=True), sequence_number=0, type='response.created')
ResponseInProgressEvent(response=Response(id='resp_0b6d7f904311f8e20069dc398817

## Structured Outputs

In [15]:
from openai import OpenAI
from pydantic import BaseModel
client = OpenAI()
class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]

response = client.responses.parse(
    model="gpt-4o-2024-08-06",
    input=[
        {"role": "system", "content": "Extract the event information."},
        {
            "role": "user",
            "content": "Alice and Bob are going to a science fair on Friday.",
        },
    ],
    text_format=CalendarEvent,
)
event = response.output_parsed
print(event)

name='Science Fair' date='Friday' participants=['Alice', 'Bob']


## Function or Tool Calling
Let’s look at an end-to-end tool calling flow for a get_horoscope function that gets a daily horoscope for an astrological sign.

In [16]:
from openai import OpenAI
import json
client = OpenAI()
# 1. Define a list of callable tools for the model
tools = [
    {
        "type": "function",
        "name": "get_horoscope",
        "description": "Get today's horoscope for an astrological sign.",
        "parameters": {
            "type": "object",
            "properties": {
                "sign": {
                    "type": "string",
                    "description": "An astrological sign like Taurus or Aquarius",
                },
            },
            "required": ["sign"],
        },
    },
]

def get_horoscope(sign):
    return f"{sign}: Next Tuesday you will befriend a baby otter."

# Create a running input list we will add to over time
input_list = [
    {"role": "user", "content": "What is my horoscope? I am an Aquarius."}
]


In [17]:
# 2. Prompt the model with tools defined
response = client.responses.create(
    model="gpt-5",
    tools=tools,
    input=input_list,
)

# Save function call outputs for subsequent requests
input_list += response.output

for item in response.output:
    if item.type == "function_call":
        if item.name == "get_horoscope":
            # 3. Execute the function logic for get_horoscope
            sign = json.loads(item.arguments)["sign"]
            horoscope = get_horoscope(sign)
            
            # 4. Provide function call results to the model
            input_list.append({
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": horoscope,
            })

print("Final input:")
print(input_list)

Final input:
[{'role': 'user', 'content': 'What is my horoscope? I am an Aquarius.'}, ResponseReasoningItem(id='rs_0fe1de083abfaf490069dc39c1e1508195991cd665298153c8', summary=[], type='reasoning', content=None, encrypted_content=None, status=None), ResponseFunctionToolCall(arguments='{"sign":"Aquarius"}', call_id='call_tMlqU08UfjteRUvGxHHeib7A', name='get_horoscope', type='function_call', id='fc_0fe1de083abfaf490069dc39c3332c81959e45cf2992bb1ba8', namespace=None, status='completed'), {'type': 'function_call_output', 'call_id': 'call_tMlqU08UfjteRUvGxHHeib7A', 'output': 'Aquarius: Next Tuesday you will befriend a baby otter.'}]


In [23]:
response = client.responses.create(
    model="gpt-5",
    instructions="Respond only with a horoscope generated by a tool.",
    tools=tools,
    input=input_list,
)

# 5. The model should be able to give a response!
print("Final output:")
print(response.model_dump_json(indent=2))
print("\n" + response.output_text)

Final output:
{
  "id": "resp_0fe1de083abfaf490069dc39e4790081959a54eaaf71f78954",
  "created_at": 1776040420.0,
  "error": null,
  "incomplete_details": null,
  "instructions": "Respond only with a horoscope generated by a tool.",
  "metadata": {},
  "model": "gpt-5-2025-08-07",
  "object": "response",
  "output": [
    {
      "id": "msg_0fe1de083abfaf490069dc39e4f6ac819589904610050e4f82",
      "content": [
        {
          "annotations": [],
          "text": "Aquarius: Next Tuesday you will befriend a baby otter.",
          "type": "output_text",
          "logprobs": []
        }
      ],
      "role": "assistant",
      "status": "completed",
      "type": "message",
      "phase": null
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [
    {
      "name": "get_horoscope",
      "parameters": {
        "type": "object",
        "properties": {
          "sign": {
            "type": "string",
            "description": "An a

### Function definition for `get_weather`

In [24]:
{
  "type": "function",
  "name": "get_weather",
  "description": "Retrieves current weather for the given location.",
  "parameters": {
    "type": "object",
    "properties": {
      "location": {
        "type": "string",
        "description": "City and country e.g. Bogotá, Colombia"
      },
      "units": {
        "type": "string",
        "enum": ["celsius", "fahrenheit"],
        "description": "Units the temperature will be returned in."
      }
    },
    "required": ["location", "units"],
    "additionalProperties": False
  },
  "strict": True
}

{'type': 'function',
 'name': 'get_weather',
 'description': 'Retrieves current weather for the given location.',
 'parameters': {'type': 'object',
  'properties': {'location': {'type': 'string',
    'description': 'City and country e.g. Bogotá, Colombia'},
   'units': {'type': 'string',
    'enum': ['celsius', 'fahrenheit'],
    'description': 'Units the temperature will be returned in.'}},
  'required': ['location', 'units'],
  'additionalProperties': False},
 'strict': True}

### Defining Namespaces
Use namespaces to group related tools by domain, such as crm, billing, or shipping. Namespaces help organize similar tools and are especially useful when the model must choose between tools that serve different systems or purposes, such as one search tool for your CRM and another for your support ticketing system.

In [26]:
{
  "type": "namespace",
  "name": "crm",
  "description": "CRM tools for customer lookup and order management.",
  "tools": [
    {
      "type": "function",
      "name": "get_customer_profile",
      "description": "Fetch a customer profile by customer ID.",
      "parameters": {
        "type": "object",
        "properties": {
          "customer_id": { "type": "string" }
        },
        "required": ["customer_id"],
        "additionalProperties": False
      }
    },
    {
      "type": "function",
      "name": "list_open_orders",
      "description": "List open orders for a customer ID.",
      "defer_loading": True,
      "parameters": {
        "type": "object",
        "properties": {
          "customer_id": { "type": "string" }
        },
        "required": ["customer_id"],
        "additionalProperties": False
      }
    }
  ]
}

{'type': 'namespace',
 'name': 'crm',
 'description': 'CRM tools for customer lookup and order management.',
 'tools': [{'type': 'function',
   'name': 'get_customer_profile',
   'description': 'Fetch a customer profile by customer ID.',
   'parameters': {'type': 'object',
    'properties': {'customer_id': {'type': 'string'}},
    'required': ['customer_id'],
    'additionalProperties': False}},
  {'type': 'function',
   'name': 'list_open_orders',
   'description': 'List open orders for a customer ID.',
   'defer_loading': True,
   'parameters': {'type': 'object',
    'properties': {'customer_id': {'type': 'string'}},
    'required': ['customer_id'],
    'additionalProperties': False}}]}

### tool_search call

In [38]:
for tool_call in response.output:
    if tool_call.type != "function_call":
        continue

    name = tool_call.name
    args = json.loads(tool_call.arguments)

    result = call_function(name, args)
    input_messages.append({
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": str(result)
    })

In [39]:
def call_function(name, args):
    if name == "get_weather":
        return get_weather(**args)
    if name == "send_email":
        return send_email(**args)

### Formatting results
### Incorporating results into response

Send results back to the model

In [41]:
input_messages = [
    {"type": "message", "role": "user", "content": "Your user prompt here"}
]

response = client.responses.create(
    model="gpt-4.1",
    input=input_messages,
    tools=tools,
)

Final response

"It's about 15°C in Paris, 18°C in Bogotá, and I've sent that email to Bob."

### Built-in Web Search
When answers depend on current information, the model retrieves and reasons over live information without custom search infrastructure.

In [42]:
response = client.responses.create(
    model=default_model,
    tools=[{"type": "web_search"}],
    input="Find two recent developments in reusable rockets.",
)
print(response.output_text)


Recent advancements in reusable rocket technology have significantly impacted the aerospace industry. Two notable developments are:

**1. SpaceX's Record-Breaking Falcon 9 Reuse**

In February 2026, SpaceX's Falcon 9 booster B1067 completed its 33rd flight, setting a new record for rocket booster reuse. This achievement underscores the company's commitment to reducing launch costs and increasing the frequency of space missions. The successful missions deployed 53 satellites, expanding SpaceX's Starlink broadband internet constellation. ([nationaltoday.com](https://nationaltoday.com/us/fl/cape-canaveral/news/2026/02/22/spacex-falcon-9-rocket-sets-new-reuse-record/?utm_source=openai))

**2. China's LandSpace Advances Reusable Rocket Technology**

Chinese private rocket manufacturer LandSpace is making significant strides in developing reusable launch vehicles. In December 2025, LandSpace's Zhuque-3 rocket successfully reached orbit, marking a significant milestone for China's commercial 

### Multimodal Input (Text + Image)

In [43]:
response = client.responses.create(
    model=default_model,
    input=[{
        "role": "user",
        "content": [
            {"type": "input_text", "text": "What is in this image?"},
            {
                "type": "input_image",
                "image_url": "https://api.nga.gov/iiif/a2e6da57-3cd1-4235-b20e-95dcaefed6c8/full/!800,800/0/default.jpg" 
            }
        ],
    }],
)
print(response.output_text)

The image depicts a portrait of a young girl sitting in a chair. She is wearing a vertically striped red and blue dress with buttons down the front, and a blue skirt with polka dots. The girl is holding a small bouquet of flowers and has a red ribbon in her hair. The background is a light, subdued color that contrasts with her vibrant clothing. The painting style is expressive, reflecting bold brush strokes typical of the post-impressionist movement.


### Background Mode for Long Tasks

In [44]:
import time
kickoff = client.responses.create(
    model=default_model,
    input="Write a detailed industry report.",
    background=True,
)
current = kickoff
while current.status not in {"completed", "failed", "cancelled", "incomplete"}:
    time.sleep(2)
    current = client.responses.retrieve(current.id)
print(current.output_text)


### Industry Report: Renewable Energy Sector

#### 1. Executive Summary

The global renewable energy sector is experiencing unprecedented growth driven by advancements in technology, policy shifts toward sustainability, and increasing consumer demand for cleaner energy sources. This report provides an overview of the current state of the renewable energy industry, key trends, market dynamics, competitive landscape, and future prospects.

#### 2. Market Overview

**2.1 Definition and Scope**
Renewable energy refers to energy obtained from natural sources that are replenished at a faster rate than they are consumed. Common types include solar, wind, hydroelectric, geothermal, and biomass energy.

**2.2 Market Size and Growth**
The global renewable energy market size was valued at approximately $1 trillion in 2022 and is projected to reach $2.15 trillion by 2030, growing at a CAGR of around 8.4%.

#### 3. Key Drivers of Growth

**3.1 Technological Advancements**
- **Solar Power**: Innovat

## Prompt Engineering
### Message `roles` and `instructions` following

In [45]:
from openai import OpenAI
client = OpenAI()

response = client.responses.create(
    model="gpt-5.4",
    reasoning={"effort": "low"},
    instructions="Talk like a pirate.",
    input="Are semicolons optional in JavaScript?",
)

print(response.output_text)

Aye — **mostly optional**, but not always wise to omit ’em.

JavaScript has a feature called **ASI** — **Automatic Semicolon Insertion**. That means the engine will often insert semicolons for ye when needed.

Example:

```js
let x = 5
let y = 10
console.log(x + y)
```

This usually works just fine.

But beware, matey: **leaving semicolons out can sometimes cause nasty surprises**. For example:

```js
return
{
  value: 42
}
```

JavaScript may treat that like:

```js
return;
{
  value: 42
}
```

So ye return `undefined` instead of the object.

Another risky case is when a line starts with `(`, `[`, or similar characters after a previous line without a semicolon.

Example:

```js
const a = 1
[1,2,3].forEach(console.log)
```

This may be parsed in an unexpected way.

## The short answer
- **Yes**, semicolons are often optional.
- **No**, they are not always safe to omit.
- **Best practice:** be consistent. Many crews either:
  - **always use semicolons**, or
  - **omit them carefully** w